# Sutlej 2025 | GMM–SLIC–Graph Cut | Version 2.1

**Publication figures: 600-DPI TIFF only** · 21 November 2026

This notebook reproduces the original 2025 Sutlej analysis and adds a compact publication-output section. Its classification settings, input rasters, water definitions, and reference-point labels remain unchanged. The original numerical findings must be checked after the notebook is rerun using the archived inputs.

**Research boundary:** The SAR and optical maps show *candidate water*. Cross-sensor agreement is not independent accuracy; the 400 Sentinel-2 RGB-interpreted labels are not field observations or a probability sample.

**Run:** Open from the project folder containing `01_Analysis_Input_Data` and `02_Analysis_Code`, or set `SUTLEJ_PROJECT_DIR`. Execute cells in order. The curated products are written separately under `03_Analysis_Outputs/07_Publication_Ready`. Source data are not bundled with this notebook. The journal's author instructions govern final figure dimensions and file formats.

**v2.1 changes:** repaired the Figure 2 bar-label overlap by placing the class labels inside panel B; curated figures are LZW-compressed 600-DPI TIFFs only. Previously exported companion PDFs in the curated folders are removed when the corresponding figure is regenerated. SAR classification and validation are unchanged.


## 1. Environment and project paths

Install the packages in `requirements.txt` if needed. No online map tiles or external credentials are required. File paths are relative to the project root.


In [ ]:
from __future__ import annotations

import json
import os
import time
import warnings
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator
from PIL import Image
from rasterio.transform import Affine, rowcol
from scipy.ndimage import label as connected_components
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import breadth_first_order, maximum_flow
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             cohen_kappa_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.mixture import GaussianMixture
from skimage.segmentation import slic

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
    'axes.labelsize': 9, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.facecolor': 'white',
})


In [ ]:
def locate_project():
    explicit = os.environ.get('SUTLEJ_PROJECT_DIR')
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not (root/'01_Analysis_Input_Data').is_dir():
            raise FileNotFoundError(f'Missing input directory under {root}')
        return root
    current = Path.cwd().resolve()
    for root in (current, *current.parents):
        if (root/'01_Analysis_Input_Data').is_dir() and (root/'02_Analysis_Code').is_dir():
            return root
    raise FileNotFoundError(
        'Start Jupyter inside Sutlej_2025_FloodMapping_Reproducibility, '
        'or set SUTLEJ_PROJECT_DIR to the project root.')

PROJECT_DIR = locate_project()
DATA_DIR = PROJECT_DIR / '01_Analysis_Input_Data'
OUTPUT_ROOT = PROJECT_DIR / '03_Analysis_Outputs'
RASTER_DIR = OUTPUT_ROOT / '01_Rasters'
MAIN_FIG_DIR = OUTPUT_ROOT / '02_Figures' / 'Manuscript'
SUPP_FIG_DIR = OUTPUT_ROOT / '02_Figures' / 'Supplementary'
MAIN_TABLE_DIR = OUTPUT_ROOT / '03_Tables' / 'Manuscript'
SUPP_TABLE_DIR = OUTPUT_ROOT / '03_Tables' / 'Supplementary'
VALIDATION_DIR = OUTPUT_ROOT / '04_Validation'
SENS_DIR = OUTPUT_ROOT / '05_Sensitivity'
REPRO_DIR = OUTPUT_ROOT / '06_Reproducibility'
for folder in (RASTER_DIR, MAIN_FIG_DIR, SUPP_FIG_DIR, MAIN_TABLE_DIR,
               SUPP_TABLE_DIR, VALIDATION_DIR, SENS_DIR, REPRO_DIR):
    folder.mkdir(parents=True, exist_ok=True)

S1_DIR = DATA_DIR / 'Sentinel_1_SAR'
S2_DIR = DATA_DIR / 'Sentinel_2_Optical'
REF_DIR = DATA_DIR / 'Reference_Data'
SAR_DRY = S1_DIR / 'S1_VV_Dry_MayJun_2025_10m.tif'
SAR_WET = S1_DIR / 'S1_VV_Wet_Sep_2025_10m.tif'
NDFI_DRY = S2_DIR / 'S2_NDFI_Dry_MayJun_2025_20m.tif'
NDFI_WET = S2_DIR / 'S2_NDFI_Wet_Sep_2025_20m.tif'
RGB_DRY = S2_DIR / 'S2_RGB_Dry_MayJun_2025_10m.tif'
RGB_WET = S2_DIR / 'S2_RGB_Wet_Sep_2025_10m.tif'
REFERENCE_SHP = REF_DIR / 'Validation_Points_2025.shp'
INPUT_FILES = (SAR_DRY, SAR_WET, NDFI_DRY, NDFI_WET,
               RGB_DRY, RGB_WET, REFERENCE_SHP)
missing = [str(p.relative_to(PROJECT_DIR)) for p in INPUT_FILES if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing required input files:\n' + '\n'.join(missing))

FIG_DPI = 600
FIG_WIDTH_MM = 180                 # Double-column, confirm with journal instructions.
EXPECTED_CRS = 'EPSG:32642'
EXPECTED_TRANSFORM_10M = Affine(10, 0, 746580, 0, -10, 3270010)
EXPECTED_TRANSFORM_20M = Affine(20, 0, 746580, 0, -20, 3270010)
AOI_HEIGHT_10M = AOI_WIDTH_10M = 2000
AOI_HEIGHT_20M = AOI_WIDTH_20M = 1000
NODATA_CODE = np.uint8(225)
ALLOW_NDFI_REPROJECT = False      # Fail on grid mismatch instead of resampling silently.
SEED = 2025                        # Sampling + block-bootstrap base seed (unchanged).
GMM_FIT_RANDOM_STATE = 42          # sklearn GaussianMixture init seed (unchanged).
GMM_N_INIT = 2                     # Number of initializations (unchanged).
GMM_MAX_ITER = 180                 # EM iteration budget (unchanged).
GMM_LOG_ODDS_LIMIT = 16.0          # Symmetric cap on pixel log odds (unchanged).
SLIC_MIN_DB = -23.0                # SAR scale lower bound (unchanged).
SLIC_MAX_DB = 2.0                  # SAR scale upper bound (unchanged).
SLIC_MAX_ITER = 7                  # SLIC iterations (unchanged).
GRAPHCUT_EDGE_SIGMA_DB = 2.0       # Pairwise contrast scale (unchanged).
GRAPHCUT_CAPACITY_SCALE = 64.0     # Float -> integer scaling (unchanged).
GMM_SAMPLES = 160_000
GMM_REG_COVAR = .025
GMM_BRIGHT_CAP_DB = -2.0
SELECTED = dict(n_segments=10_000, compactness=.08,
                graphcut_lambda=10.0, min_patch_pixels_10m=100)
WATER_VOTES_PER_2X2 = 3
RUN_SAR_SENSITIVITY = True
SENSITIVITY_VARIANTS = [
    (6400,.08,10),(10000,.08,0),(10000,.08,2),
    (10000,.08,5),(10000,.08,10),(10000,.08,20),
    (16000,.08,10),(10000,.03,10),(10000,.16,10),
]
NDFI_MARGIN_SCENARIOS = [(0.,0.),(.03,.10),(.05,.15),(.10,.20)]
RUN_KAPPA_BLOCK_BOOTSTRAP = True
BOOTSTRAP_BLOCK_SIZE_20M = 50
BOOTSTRAP_REPLICATES = 300

sar_path = SAR_WET
sar_dry_path = SAR_DRY
ndfi_path = NDFI_WET
dry_ndfi_path = NDFI_DRY
print('Project:', PROJECT_DIR)
print('Reference points:', REFERENCE_SHP.relative_to(PROJECT_DIR))
print('Results:', OUTPUT_ROOT.relative_to(PROJECT_DIR))


## 2. Sentinel-1 wet VV: input checks and GMM

Dry and wet VV are separate, single-band files. Their grids must match. Classification uses **wet VV only**; the dry file is checked for consistency but does not enter the fitted model.


In [ ]:
with rasterio.open(sar_dry_path) as dry_ds, rasterio.open(sar_path) as ds:
    for source in (dry_ds, ds):
        if source.count != 1 or source.crs is None or source.crs.to_string() != EXPECTED_CRS:
            raise ValueError(f'Expected one-band EPSG:32642 SAR: {source.name}')
        if source.width < 2000 or source.height < 2000:
            raise ValueError(f'SAR raster smaller than the study window: {source.name}')
        if not np.allclose(tuple(source.transform)[:6],
                           tuple(EXPECTED_TRANSFORM_10M)[:6],rtol=0,atol=1e-8):
            raise ValueError(f'Incorrect 10-m alignment: {source.name}')
    if dry_ds.transform != ds.transform or (dry_ds.width, dry_ds.height) != (ds.width, ds.height):
        raise ValueError('Dry and wet SAR files must have the same raster grid and size')
    window = rasterio.windows.Window(0, 0, 2000, 2000)
    wet_vv = ds.read(1,window=window).astype(np.float32)
    wet_mask = ds.read_masks(1,window=window) > 0
    transform_10m = ds.window_transform(window)
    sar_crs = ds.crs

wet_valid = wet_mask & np.isfinite(wet_vv)
if not np.any(wet_valid):
    raise ValueError('No valid wet-season SAR pixels')
transform_20m = transform_10m * Affine.scale(2,2)
assert np.allclose(tuple(transform_20m)[:6],
                   tuple(EXPECTED_TRANSFORM_20M)[:6],rtol=0,atol=1e-8)
wet_values = wet_vv[wet_valid]
wet_stats = pd.Series({
    'valid_pixels': int(wet_valid.sum()),
    'valid_pct': float(100*wet_valid.mean()),
    'mean_dB': float(wet_values.mean()),
    'median_dB': float(np.median(wet_values)),
    'p05_dB': float(np.percentile(wet_values,5)),
    'p95_dB': float(np.percentile(wet_values,95)),
}, name='Wet-season VV')
wet_stats.to_frame().to_csv(SUPP_TABLE_DIR/'Table_S01_Wet_VV_Statistics.csv')


In [ ]:
# The random subsample seed and the GMM initialization seed are DISTINCT.
# Preserve both, because conflating them makes the analysis difficult to reproduce.
rng = np.random.default_rng(SEED)
training = rng.choice(wet_values, size=min(GMM_SAMPLES, wet_values.size), replace=False).reshape(-1, 1)
if not np.isfinite(training).all():
    raise ValueError('GMM training contains non-finite wet-season VV samples')

gmm = GaussianMixture(n_components=2, n_init=GMM_N_INIT,
                      max_iter=GMM_MAX_ITER, reg_covar=GMM_REG_COVAR,
                      random_state=GMM_FIT_RANDOM_STATE)
gmm.fit(training)
if not gmm.converged_:
    raise RuntimeError('GMM did not converge. Investigate inputs/fit; do not publish this run.')
means = gmm.means_.ravel()
sigmas = np.sqrt(gmm.covariances_.reshape(-1))
priors = gmm.weights_.ravel()
if not (np.isfinite(means).all() and np.isfinite(sigmas).all()
        and np.isfinite(priors).all() and (sigmas > 0).all() and (priors > 0).all()):
    raise RuntimeError('GMM component parameters are invalid')
water_component, background_component = np.argsort(means)
separation_sigma = float((means[background_component] - means[water_component]) /
                         np.sqrt(0.5 * (sigmas[water_component]**2 + sigmas[background_component]**2)))
if separation_sigma < 1.0:
    warnings.warn('GMM low/high VV components overlap strongly; candidate-water separation is weak.')

gmm_table = pd.DataFrame({
    'component': ['low-VV candidate water' if i == water_component else 'higher-VV background' for i in range(2)],
    'mean_dB': means, 'std_dB': sigmas, 'prior': priors,
}).sort_values('mean_dB')
gmm_table.to_csv(SUPP_TABLE_DIR/'Table_S02_GMM_Components.csv', index=False)

from scipy.stats import norm

# Unequal-variance Gaussian posterior odds can cross ZERO twice. For publication,
# report the implied scene-specific pixel-level boundary rather than claiming
# that GMM completely avoids a decision threshold. The > -2 dB tail is capped.
lo, hi = int(water_component), int(background_component)
a = -0.5 * (1/sigmas[lo]**2 - 1/sigmas[hi]**2)
b = means[lo]/sigmas[lo]**2 - means[hi]/sigmas[hi]**2
c = (np.log(priors[lo]/priors[hi]) + np.log(sigmas[hi]/sigmas[lo])
     - 0.5*(means[lo]**2/sigmas[lo]**2 - means[hi]**2/sigmas[hi]**2))
coefficients = [a, b, c] if abs(a) > 1e-12 else [b, c]
real_crossings = sorted(float(root.real) for root in np.roots(coefficients)
                        if abs(root.imag) < 1e-8) if abs(a) > 1e-12 or abs(b) > 1e-12 else []
# Only crossings inside the observed valid backscatter domain and at/below the
# bright cap are realized pixel-level boundary candidates in this raster.
observed_min_vv = float(np.min(wet_values))
observed_max_capped_vv = float(min(np.max(wet_values), GMM_BRIGHT_CAP_DB))
operative_crossings = [x for x in real_crossings
                       if observed_min_vv <= x <= observed_max_capped_vv]
gmm_diagnostics = dict(
    converged=bool(gmm.converged_), em_iterations=int(gmm.n_iter_),
    sample_seed=int(SEED), fit_random_state=int(GMM_FIT_RANDOM_STATE),
    n_init=int(GMM_N_INIT), requested_sample_size=int(GMM_SAMPLES),
    actual_sample_size=int(len(training)),
    component_separation_pooled_sigma=separation_sigma,
    all_posterior_crossings_db=real_crossings,
    posterior_crossings_at_or_below_bright_cap_db=operative_crossings,
    observed_valid_min_db=observed_min_vv,
    observed_max_after_bright_cap_db=observed_max_capped_vv,
    bright_cap_db=float(GMM_BRIGHT_CAP_DB),
    log_odds_limit=float(GMM_LOG_ODDS_LIMIT),
)
(SUPP_TABLE_DIR/'GMM_Fit_Diagnostics.json').write_text(
    json.dumps(gmm_diagnostics, indent=2), encoding='utf-8')
print('GMM fit diagnostics:', json.dumps(gmm_diagnostics, indent=2))

# Do not evaluate the PDFs on masked NaN/Inf pixels; valid-pixel math is
# IDENTICAL to the original baseline and masked log odds remain zero.
evaluation_vv = np.minimum(np.where(wet_valid, wet_vv, GMM_BRIGHT_CAP_DB),
                           GMM_BRIGHT_CAP_DB)
log_low = np.log(priors[lo]) + norm.logpdf(evaluation_vv, means[lo], sigmas[lo])
log_high = np.log(priors[hi]) + norm.logpdf(evaluation_vv, means[hi], sigmas[hi])
log_odds = np.clip(log_low - log_high, -GMM_LOG_ODDS_LIMIT,
                   GMM_LOG_ODDS_LIMIT).astype(np.float32)
log_odds[~wet_valid] = 0.0
assert np.isfinite(log_odds).all(), 'Non-finite graph-cut unary odds'


## 3. SLIC segmentation and graph-cut regularization

The original GMM, SLIC, graph-cut, and patch-filter equations and baseline parameters are retained. SAR sensitivity analyses do not use optical maps or reference labels for model selection.


**Review note:** This uses a *scene-fitted* pixel posterior preference and a graph-cut spatial regularizer. The pixel odds can imply a backscatter decision boundary, and the integer-capacity minimum cut is exact only for its quantized graph energy. The λ scale depends on superpixel sizes; the SAR-only sensitivity table supports interpretation but is not a ground-truth optimization.


In [ ]:
def slic_segments(wet: np.ndarray, valid: np.ndarray, n_segments: int,
                  compactness: float) -> np.ndarray:
    # Spatial SLIC in intensity+XY space. Missing pixels filled only for segmentation.
    if wet.shape != valid.shape or wet.ndim != 2 or not np.any(valid):
        raise ValueError('SLIC expects matching 2D wet/valid arrays with valid pixels')
    if not np.isfinite(wet[valid]).all():
        raise ValueError('Valid SLIC wet VV pixels must be finite')
    if int(n_segments) < 2 or not np.isfinite(compactness) or compactness <= 0:
        raise ValueError('SLIC n_segments must be >= 2 and compactness positive')
    from scipy.ndimage import distance_transform_edt
    if not valid.all():
        # Nearest finite neighbor purely for computational segmentation; output keeps NoData.
        idx = distance_transform_edt(~valid, return_distances=False, return_indices=True)
        filled = wet[tuple(idx)]
    else:
        filled = wet
    scaled = np.clip((filled - SLIC_MIN_DB) / (SLIC_MAX_DB - SLIC_MIN_DB),
                     0, 1).astype(np.float32)
    segments = slic(scaled, n_segments=int(n_segments),compactness=float(compactness),
                    sigma=0,channel_axis=None,start_label=0,
                    enforce_connectivity=True,max_num_iter=SLIC_MAX_ITER)
    return segments.astype(np.int32)


def graph_cut_water(segments: np.ndarray, wet: np.ndarray, valid: np.ndarray,
                    wet_log_odds: np.ndarray, lam: float,
                    edge_sigma_db: float = GRAPHCUT_EDGE_SIGMA_DB,
                    capacity_scale: float = GRAPHCUT_CAPACITY_SCALE):
    # Exact integer-capacity binary s-t min-cut over SLIC nodes.
    # For the unquantized real-valued energy, the 64x integer scaling and
    # rounding introduce small discretization; do not call it an exact real cut.
    if not (segments.shape == wet.shape == valid.shape == wet_log_odds.shape):
        raise ValueError('Graph-cut arrays must have identical dimensions')
    if segments.ndim != 2 or not valid.any() or int(segments.min()) < 0:
        raise ValueError('Graph cut requires 2D nonnegative labels and valid pixels')
    if not (np.isfinite(lam) and lam >= 0 and np.isfinite(edge_sigma_db)
            and edge_sigma_db > 0 and np.isfinite(capacity_scale) and capacity_scale > 0):
        raise ValueError('Graph-cut lambda must be >= 0, scales must be positive')
    if not (np.isfinite(wet[valid]).all() and np.isfinite(wet_log_odds[valid]).all()):
        raise ValueError('Graph-cut valid pixels must have finite VV and log odds')
    n = int(segments.max())+1
    flat_segments = segments.ravel()
    valid_flat = valid.ravel()
    counts = np.bincount(flat_segments[valid_flat],minlength=n)
    sums = np.bincount(flat_segments[valid_flat],weights=wet.ravel()[valid_flat],minlength=n)
    means = np.divide(sums,counts,out=np.zeros(n,dtype=np.float64),where=(counts>0))
    unary_log_odds = np.bincount(flat_segments[valid_flat],
                                  weights=wet_log_odds.ravel()[valid_flat],minlength=n)
    # Boundary length in pixel edges for each UNIQUE undirected adjacent pair.
    keys=[]
    for first,second,pair_valid in ((segments[:,1:],segments[:,:-1], valid[:,1:] & valid[:,:-1]),
                                    (segments[1:,:],segments[:-1,:], valid[1:,:] & valid[:-1,:])):
        differing = (first!=second) & pair_valid
        a=first[differing].astype(np.int64)
        b=second[differing].astype(np.int64)
        keys.append(np.minimum(a,b)*n+np.maximum(a,b))
    unique, shared_length = np.unique(np.concatenate(keys),return_counts=True)
    a=(unique//n).astype(np.int32)
    b=(unique%n).astype(np.int32)
    edge_cost = float(lam)*shared_length*np.exp(-0.5*((means[a]-means[b])/edge_sigma_db)**2)
    # For positive log odds, source-water attachment is favorable.
    source = n
    sink = n+1
    indices = np.concatenate([a,b,np.full(n,source,dtype=np.int32),np.arange(n,dtype=np.int32)])
    targets = np.concatenate([b,a,np.arange(n,dtype=np.int32),np.full(n,sink,dtype=np.int32)])
    unrounded_capacities = np.concatenate([edge_cost, edge_cost,
                            np.maximum(unary_log_odds,0), np.maximum(-unary_log_odds,0)]) * capacity_scale
    if not np.isfinite(unrounded_capacities).all() or (unrounded_capacities < 0).any():
        raise ValueError('Graph-cut capacities must be finite and nonnegative')
    if unrounded_capacities.size and unrounded_capacities.max() >= np.iinfo(np.int64).max:
        raise OverflowError('Graph-cut capacity exceeds int64 range')
    capacities = np.rint(unrounded_capacities).astype(np.int64)
    graph = coo_matrix((capacities,(indices,targets)),shape=(n+2,n+2)).tocsr()
    result = maximum_flow(graph,source,sink,method='dinic')
    residual = graph-result.flow
    reachable = breadth_first_order((residual>0).tocsr(),source,
                                    directed=True,return_predecessors=False)
    labels = np.zeros(n,dtype=np.uint8)
    labels[reachable[reachable<n]] = 1
    pixel_water = (labels[segments] == 1) & valid
    local_rms = np.sqrt(np.mean((wet[valid]-means[segments[valid]])**2))
    nonempty = counts[counts > 0]
    return pixel_water, {
        'segments_actual': n, 'segments_with_valid_pixels': int(len(nonempty)),
        'median_valid_pixels_per_segment': float(np.median(nonempty)),
        'slic_within_segment_rms_db': float(local_rms),
        'graph_edges': int(len(unique)),
        'graphcut_unary_abs_sum': float(np.abs(unary_log_odds).sum()),
        'graphcut_pairwise_cost_sum_at_lambda': float(edge_cost.sum()),
        'graphcut_max_flow_scaled': int(result.flow_value),
        'graphcut_capacity_scale': float(capacity_scale),
        'graphcut_pairwise_sigma_db': float(edge_sigma_db),
        'water_pct_raw_valid': float(100 * pixel_water.sum() / valid.sum()),
    }


def remove_small_water_patches(water: np.ndarray, valid: np.ndarray,
                               min_pixels: int=100):
    # 8-connected removal; removed valid water pixels become 0 (not NoData).
    regions,nregions=connected_components(water & valid,structure=np.ones((3,3),dtype=np.uint8))
    sizes=np.bincount(regions.ravel())
    rejected=(sizes<min_pixels)
    rejected[0]=False
    water_clean=water & ~rejected[regions] & valid
    return water_clean,{'number_of_removed_patches':int(rejected.sum()),
                        'removed_water_pixels_10m':int((water & ~water_clean).sum())}


def classify_sar(n_segments:int,compactness:float,lam:float,minimum:int,
                 return_segments:bool=False, precomputed_segments=None):
    t0=time.time()
    # Reuse the already-computed baseline SLIC map from the sensitivity sweep.
    # This avoids a redundant 2000x2000 SLIC run; the labels are unchanged.
    segments = (precomputed_segments if precomputed_segments is not None else
                slic_segments(wet_vv,wet_valid,n_segments,compactness))
    if precomputed_segments is not None and segments.shape != wet_vv.shape:
        raise ValueError('Cached SLIC segments have incorrect shape')
    rough,diagnostics=graph_cut_water(segments,wet_vv,wet_valid,log_odds,lam)
    clean,patch_info=remove_small_water_patches(rough,wet_valid,minimum)
    diagnostics.update(patch_info)
    diagnostics.update(n_segments_requested=int(n_segments),compactness=float(compactness),
                       graphcut_lambda=float(lam),min_patch_pixels_10m=int(minimum),
                       water_pct_clean_valid=float(clean.sum()/wet_valid.sum()*100),
                       elapsed_seconds=float(time.time()-t0))
    if return_segments:
        return clean,diagnostics,segments
    return clean,diagnostics


In [ ]:
sensitivity_records=[]

if RUN_SAR_SENSITIVITY:
    # Classification masks need about 4 MB per variant; cache for baseline IoU.
    candidate_masks=[]
    segments_cache={}
    selected_segments=None
    selected_segments_key=None
    for number,compact,lam in SENSITIVITY_VARIANTS:
        key=(number,compact)
        if key not in segments_cache:
            segments_cache.clear()
            segments_cache[key]=slic_segments(wet_vv,wet_valid,number,compact)
        labels_now=segments_cache[key]
        if key == (SELECTED['n_segments'], SELECTED['compactness']):
            selected_segments=labels_now
            selected_segments_key=key
        raw,detail=graph_cut_water(labels_now,wet_vv,wet_valid,log_odds,lam)
        cleaned,removed=remove_small_water_patches(raw,wet_valid,SELECTED['min_patch_pixels_10m'])
        detail.update(removed)
        for patch_threshold in (25,50,100,200,400):
            candidate,_=remove_small_water_patches(raw,wet_valid,patch_threshold)
            detail[f'water_km2_patch_min_{patch_threshold}px'] = float(candidate.sum()*.0001)
        detail.update(n_segments_requested=number, compactness=compact,
                      graphcut_lambda=lam,
                      water_pct_clean_valid=float(100*cleaned.sum()/wet_valid.sum()),
                      water_area_km2_10m=float(cleaned.sum()*.0001))
        candidate_masks.append(cleaned)
        sensitivity_records.append(detail)
    segments_cache.clear()
    baseline_key=(SELECTED['n_segments'],SELECTED['compactness'],SELECTED['graphcut_lambda'])
    try:
        baseline_index=SENSITIVITY_VARIANTS.index(baseline_key)
    except ValueError:
        raise ValueError('SELECTED SLIC and Graph Cut parameters must appear in SENSITIVITY_VARIANTS')
    if selected_segments is None:
        raise RuntimeError('Baseline SLIC segmentation was not found in sensitivity variants')
    baseline=candidate_masks[baseline_index]
    baseline_area=int(baseline.sum())
    for ix,mask in enumerate(candidate_masks):
        overlap=np.count_nonzero(mask & baseline)
        union=np.count_nonzero(mask | baseline)
        sensitivity_records[ix]['water_IoU_vs_baseline_10m']=overlap/union if union else np.nan
        sensitivity_records[ix]['area_difference_vs_baseline_km2']=(int(mask.sum())-baseline_area)*.0001
    del candidate_masks,baseline
    sensitivity=pd.DataFrame(sensitivity_records)
    sensitivity.to_csv(SENS_DIR/'Table_S03_SAR_Parameter_Sensitivity.csv',index=False)
    display(sensitivity.round(3))
    print('Saved SAR sensitivity data.')
else:
    sensitivity=pd.DataFrame()
    selected_segments=None
    selected_segments_key=None
    print('SAR sensitivity disabled.')


## 4. SAR candidate water at 10 m and 20 m

Water is assigned to a 20-m cell when at least 3 of 4 valid 10-m pixels are water. NoData remains 225. Outputs are georeferenced GeoTIFFs, not manuscript figures.


In [ ]:
sar_clean_10m, sar_info, final_segments = classify_sar(
    SELECTED['n_segments'],SELECTED['compactness'],
    SELECTED['graphcut_lambda'],SELECTED['min_patch_pixels_10m'],
    return_segments=True,
    precomputed_segments=(selected_segments if selected_segments_key ==
                          (SELECTED['n_segments'], SELECTED['compactness']) else None))
print(json.dumps(sar_info,indent=2))

# Save native 10-m SAR classification as an auditable georeferenced product.
sar_10m_classes = np.full(wet_vv.shape,NODATA_CODE,dtype=np.uint8)
sar_10m_classes[wet_valid] = sar_clean_10m[wet_valid].astype(np.uint8)
sar_output_10m = RASTER_DIR/'Sutlej2025_SAR_WetWater_GMM_SLIC_GraphCut_10m.tif'
with rasterio.open(sar_output_10m,'w',driver='GTiff',height=2000,width=2000,
                   count=1,dtype='uint8',nodata=225,crs=sar_crs,
                   transform=transform_10m,compress='deflate',predictor=1) as dst:
    dst.write(sar_10m_classes,1)
    dst.set_band_description(1,'Wet_season_SAR_candidate_water')


def aggregate_binary_10_to_20(water,valid,water_votes=3):
    if water_votes not in (1,2,3,4):
        raise ValueError('water_votes must be 1,2,3 or 4')
    h,w=water.shape
    if h%2 or w%2:
        raise ValueError('Input to 2×2 aggregation must have even dimensions')
    count_valid=valid.reshape(h//2,2,w//2,2).sum(axis=(1,3))
    count_water=(water & valid).reshape(h//2,2,w//2,2).sum(axis=(1,3))
    out=np.full((h//2,w//2), NODATA_CODE,dtype=np.uint8)
    # Strict validity: if even one 10-m child is missing, mark the 20-m cell NoData.
    # This intentionally preserves the original baseline semantics.
    full=(count_valid==4)
    out[full]=np.where(count_water[full]>=water_votes,1,0).astype(np.uint8)
    return out

sar_binary_20m = aggregate_binary_10_to_20(sar_clean_10m,wet_valid,WATER_VOTES_PER_2X2)

# SAR-only aggregation sensitivity, not tuned to NDFI.
vote_sensitivity=pd.DataFrame([{'water_votes_required':votes,
    'water_km2':np.count_nonzero(aggregate_binary_10_to_20(sar_clean_10m,wet_valid,votes)==1)*.0004}
    for votes in (2,3,4)])
display(vote_sensitivity)
vote_sensitivity.to_csv(SENS_DIR/'SAR_20m_vote_sensitivity.csv',index=False)
assert sar_binary_20m.shape == (AOI_HEIGHT_20M,AOI_WIDTH_20M)
print('SAR 20m class counts:', dict(zip(*[x.tolist() for x in np.unique(sar_binary_20m,return_counts=True)])))

def save_u8_map(path,arr,description,tags=None):
    assert arr.dtype==np.uint8
    assert arr.shape==(AOI_HEIGHT_20M,AOI_WIDTH_20M)
    assert set(np.unique(arr)).issubset({0,1,2,3,4,225}), 'Unexpected class code'
    profile=dict(driver='GTiff',height=arr.shape[0],width=arr.shape[1],count=1,
                 dtype='uint8',nodata=int(NODATA_CODE),crs=sar_crs,
                 transform=transform_20m,compress='deflate',predictor=1)
    with rasterio.open(path,'w',**profile) as dst:
        dst.write(arr,1)
        dst.set_band_description(1,description)
        dst.update_tags(**(tags or {}))
    return path

sar_output = save_u8_map(
    RASTER_DIR/'Sutlej2025_WetVV_GMM_SLIC_GraphCut_Water_20m.tif',
    sar_binary_20m,'SAR_wet_candidate_water',
    {'input_band':'Single-band wet-season VV','water_code':'1','nonwater_code':'0','nodata_code':'225',
     'wet_only':'true','thresholding':'GMM + SLIC + graph cut',
     'postprocessing':f'8-connected component removal, then {WATER_VOTES_PER_2X2}-of-4 vote',
     'interpretation':'wet-season candidate water; evaluated with reference points downstream'}
)
print('Saved:',sar_output)


## 5. Sentinel-2 NDFI and wet-season water

The **continuous** dry and wet NDFI GeoTIFFs are evaluated on the original 20-m grid. Wet-season optical water is `NDFI ≥ 0`. Neither sensor is treated as truth when measuring inter-method agreement.


In [ ]:
def read_ndfi_on_20m_grid(path: Path):
    with rasterio.open(path) as ds:
        if ds.count != 1:
            raise ValueError(f'Expected one-band continuous NDFI in {path.name}')
        aligned = (ds.crs == sar_crs and np.allclose(tuple(ds.transform)[:6],
                   tuple(transform_20m)[:6],rtol=0,atol=1e-8))
        source_info = dict(filename=path.name, source_shape=[ds.height,ds.width],
            source_crs=str(ds.crs), source_transform=list(tuple(ds.transform)[:6]),
            source_nodata=ds.nodata)
        if aligned and ds.width >= 1000 and ds.height >= 1000:
            # A 1001x1001 export extends one pixel outside the 1000x1000 AOI.
            arr = ds.read(1,window=rasterio.windows.Window(0,0,1000,1000),masked=True)
            data = arr.filled(np.nan).astype(np.float32)
            good = (~np.ma.getmaskarray(arr)) & np.isfinite(data)
            source_info['alignment'] = 'EXACT GRID: cropped extra edge; no resampling'
        elif ALLOW_NDFI_REPROJECT:
            if ds.crs is None: raise ValueError('Missing NDFI source CRS')
            source = ds.read(1,masked=True).filled(np.nan).astype(np.float32)
            data = np.full((1000,1000),np.nan,dtype=np.float32)
            reproject(source=source,destination=data,src_transform=ds.transform,
                src_crs=ds.crs,src_nodata=np.nan,dst_transform=transform_20m,
                dst_crs=sar_crs,dst_nodata=np.nan,resampling=Resampling.nearest)
            good=np.isfinite(data)
            source_info['alignment']='EXPLICIT nearest-neighbor reprojection'
            warnings.warn('NDFI was reprojected; inspect grid and boundary effects.')
        else:
            raise ValueError(f'NDFI source grid does not match the study grid: {path.name}; '
                             f'CRS={ds.crs}, transform={ds.transform}. '
                             'Inspect before optionally allowing reprojection.')
    if np.any(good & ((data < -1.05)|(data > 1.05))):
        warnings.warn(f'{path.name}: NDFI outside [-1.05,1.05]; inspect scaling/NoData')
    return data,good,source_info

ndfi_wet_20m, ndfi_valid_20m,wet_ndfi_info=read_ndfi_on_20m_grid(ndfi_path)
ndfi_dry_20m, ndfi_dry_valid_20m,dry_ndfi_info=read_ndfi_on_20m_grid(dry_ndfi_path)
ndfi_joint_valid_20m=ndfi_valid_20m & ndfi_dry_valid_20m
ndfi_delta_20m=np.full((1000,1000),np.nan,dtype=np.float32)
ndfi_delta_20m[ndfi_joint_valid_20m]=(ndfi_wet_20m[ndfi_joint_valid_20m]-
                                      ndfi_dry_20m[ndfi_joint_valid_20m])
print('Wet NDFI:',wet_ndfi_info)
print('Dry NDFI:',dry_ndfi_info)

ndfi_stats=[]
for name,arr,valid in [('dry NDFI',ndfi_dry_20m,ndfi_dry_valid_20m),
                       ('wet NDFI',ndfi_wet_20m,ndfi_valid_20m),
                       ('wet − dry NDFI',ndfi_delta_20m,ndfi_joint_valid_20m)]:
    v=arr[valid]
    if not v.size: raise ValueError(f'No valid pixels in {name}')
    ndfi_stats.append(dict(dataset=name,valid_pixels=int(v.size),
        minimum=float(v.min()),p01=float(np.percentile(v,1)),
        p05=float(np.percentile(v,5)),p25=float(np.percentile(v,25)),
        median=float(np.median(v)),mean=float(v.mean()),
        p75=float(np.percentile(v,75)),p95=float(np.percentile(v,95)),
        p99=float(np.percentile(v,99)),maximum=float(v.max())))
ndfi_stats_table=pd.DataFrame(ndfi_stats)
ndfi_stats_table.to_csv(SUPP_TABLE_DIR/'NDFI_continuous_dry_wet_difference_stats.csv',index=False)
display(ndfi_stats_table.round(4))

# Wet NDFI categorical map: 1=wet water, 0=wet non-water, 225=missing.
ndfi_binary_20m=np.full((1000,1000),NODATA_CODE,dtype=np.uint8)
ndfi_binary_20m[ndfi_valid_20m]=(ndfi_wet_20m[ndfi_valid_20m]>=0).astype(np.uint8)
ndfi_output=save_u8_map(RASTER_DIR/'Sutlej2025_NDFI_WetWater_20m.tif',
                        ndfi_binary_20m,'NDFI_wet_candidate_water',
                        {'water_threshold':'wet NDFI >= 0','no_water_mask_applied':'true',
                         'water_code':'1','non_water_code':'0','nodata_code':'225'})
print('Saved wet NDFI binary:',ndfi_output)


## 6. Wet-season inter-method agreement

Four classes distinguish water detected by both sources, SAR alone, NDFI alone, or neither. Cohen's κ and intersection-over-union describe **agreement**, not reference accuracy.


In [ ]:
valid_pair=np.isin(sar_binary_20m,[0,1]) & np.isin(ndfi_binary_20m,[0,1])
assert valid_pair.any(),'No common valid pixels'
sv=sar_binary_20m[valid_pair]
nv=ndfi_binary_20m[valid_pair]
confusion=np.zeros((2,2),dtype=np.int64)
np.add.at(confusion,(sv,nv),1)
# rows: SAR 0,1; columns: NDFI 0,1.
both_water=int(confusion[1,1]); sar_only=int(confusion[1,0])
optical_only=int(confusion[0,1]); neither=int(confusion[0,0])
n_compared=int(confusion.sum())
observed_agreement=(both_water+neither)/n_compared
chance_agreement=(confusion.sum(axis=0)*confusion.sum(axis=1)).sum()/n_compared**2
kappa=(observed_agreement-chance_agreement)/(1-chance_agreement) if chance_agreement < 1 else np.nan
water_iou=both_water/(both_water+sar_only+optical_only) if both_water+sar_only+optical_only else np.nan
water_dice=(2*both_water)/(2*both_water+sar_only+optical_only) if 2*both_water+sar_only+optical_only else np.nan
agreement_metrics={
    'comparison':'inter-method wet-season water agreement; NOT accuracy',
    'compared_pixels':n_compared,'compared_area_km2':n_compared*.0004,
    'observed_agreement':observed_agreement,'expected_chance_agreement':chance_agreement,
    'cohens_kappa':kappa,'water_iou':water_iou,'water_dice':water_dice,
    'both_water_km2':both_water*.0004,'SAR_only_water_km2':sar_only*.0004,
    'optical_only_water_km2':optical_only*.0004,'neither_water_km2':neither*.0004,
    'missing_either_pixels':int((~valid_pair).sum()),
    'SAR_water_area_whole_grid_km2':int((sar_binary_20m==1).sum())*.0004,
    'wet_NDFI_water_area_whole_grid_km2':int((ndfi_binary_20m==1).sum())*.0004,
}
confusion_df=pd.DataFrame(confusion,index=['SAR non-water','SAR water'],
                           columns=['NDFI non-water','NDFI water'])
display(confusion_df)
display(pd.Series(agreement_metrics,name='Wet-water inter-method agreement').to_frame())

# Four categories + one explicit NoData code.
fusion_4class=np.full((1000,1000),NODATA_CODE,dtype=np.uint8)
fusion_4class[valid_pair & (sar_binary_20m==0)&(ndfi_binary_20m==0)]=0
fusion_4class[valid_pair & (sar_binary_20m==1)&(ndfi_binary_20m==1)]=1
fusion_4class[valid_pair & (sar_binary_20m==1)&(ndfi_binary_20m==0)]=2
fusion_4class[valid_pair & (sar_binary_20m==0)&(ndfi_binary_20m==1)]=3
assert int(np.sum(fusion_4class==1))==both_water
assert int(np.sum(fusion_4class==2))==sar_only
assert int(np.sum(fusion_4class==3))==optical_only
assert int(np.sum(fusion_4class==0))==neither
fusion_output=save_u8_map(RASTER_DIR/'Sutlej2025_WetSAR_WetNDFI_Fusion4Class_20m.tif',
    fusion_4class,'wetSAR_wetNDFI_4class_fusion',
    {'0':'D_neither','1':'A_both_water','2':'C_SAR_only','3':'B_optical_only',
     '225':'NoData','analysis':'wet-water inter-method agreement; not accuracy'})
fusion_areas=pd.DataFrame([dict(code=i,category=meaning,pixels=int((fusion_4class==i).sum()),
    area_km2=float((fusion_4class==i).sum())*.0004)
    for i,meaning in [(0,'D: neither water'),(1,'A: both water'),
                      (2,'C: SAR only'),(3,'B: optical only'),(225,'NoData')]])
fusion_areas.to_csv(SUPP_TABLE_DIR/'WetSAR_WetNDFI_Fusion4Class_Areas.csv',index=False)
confusion_df.to_csv(SUPP_TABLE_DIR/'WetSAR_WetNDFI_confusion_matrix.csv')
pd.Series(agreement_metrics).to_csv(SUPP_TABLE_DIR/'WetSAR_WetNDFI_agreement_metrics.csv',header=['value'])
display(fusion_areas)
print('4-class fusion raster:',fusion_output)

# Optional: approximate spatial-block uncertainty in *agreement*, not truth accuracy.
if RUN_KAPPA_BLOCK_BOOTSTRAP:
    width=BOOTSTRAP_BLOCK_SIZE_20M
    if 1000%width:raise ValueError('Bootstrap block width must divide 1000')
    blocks=np.zeros(((1000//width)**2,4),dtype=np.int64)
    for s in (0,1):
        for o in (0,1):
            pair=valid_pair & (sar_binary_20m==s)&(ndfi_binary_20m==o)
            blocks[:,s*2+o]=pair.reshape(1000//width,width,1000//width,width).sum(axis=(1,3)).ravel()
    random_generator=np.random.default_rng(SEED+44)
    bootstrap_kappas=[]
    for _ in range(BOOTSTRAP_REPLICATES):
        idx=random_generator.integers(0,len(blocks),size=len(blocks))
        counts=blocks[idx].sum(axis=0).reshape(2,2)
        total=counts.sum()
        if not total:continue
        obs=np.trace(counts)/total
        exp=(counts.sum(axis=0)*counts.sum(axis=1)).sum()/total**2
        if exp<1:bootstrap_kappas.append((obs-exp)/(1-exp))
    if bootstrap_kappas:
        confidence_interval=np.quantile(bootstrap_kappas,[.025,.975])
        agreement_metrics.update(kappa_block_bootstrap_95pct_low=float(confidence_interval[0]),
            kappa_block_bootstrap_95pct_high=float(confidence_interval[1]),
            kappa_bootstrap_block_width_m=int(width*20),
            kappa_bootstrap_replicates=len(bootstrap_kappas))
        pd.Series(agreement_metrics).to_csv(SUPP_TABLE_DIR/'WetSAR_WetNDFI_agreement_metrics.csv',header=['value'])
        print('Approx. spatial-block Kappa 95% interval:',confidence_interval,
              '(agreement uncertainty only, NOT classification accuracy)')


## 7. Optical change and corroborated candidate inundation

New optical water requires **dry NDFI < 0 and wet NDFI ≥ 0**. Corroborated candidate inundation additionally requires wet-season SAR water. It is not a SAR dry-to-wet change detector.


In [ ]:
d=ndfi_dry_20m; w=ndfi_wet_20m; valid_change=ndfi_joint_valid_20m
ndfi_transition=np.full((1000,1000),NODATA_CODE,dtype=np.uint8)
ndfi_transition[valid_change & (d<0) &(w<0)]=0
ndfi_transition[valid_change & (d<0) &(w>=0)]=1
ndfi_transition[valid_change & (d>=0)&(w>=0)]=2
ndfi_transition[valid_change & (d>=0)&(w<0)]=3
assert np.all(ndfi_transition[valid_change]!=NODATA_CODE)
transition_names={0:'Stable non-water',1:'Newly appearing water candidate',
                  2:'Water present in both periods',3:'Water-to-non-water transition',
                  225:'At least one NDFI observation missing'}
transition_areas=pd.DataFrame([dict(code=code,meaning=label,
    pixels=int((ndfi_transition==code).sum()),area_km2=float((ndfi_transition==code).sum())*.0004)
    for code,label in transition_names.items()])
transition_areas.to_csv(SUPP_TABLE_DIR/'NDFI_DryWet_ZeroThreshold_TransitionAreas.csv',index=False)
display(transition_areas)
transition_output=save_u8_map(RASTER_DIR/'Sutlej2025_NDFI_DryWet_Transitions4Class_20m.tif',
    ndfi_transition,'NDFI_dry_to_wet_transition',
    {'classes':'0 stable nonwater,1 new-water candidate,2 persistent water,3 receded water,225 NoData',
     'rule':'dry<0 AND wet>=0 for new water','analysis':'optical change; not validation'})

def save_f32_map(path,array,description):
    assert array.shape==(1000,1000)
    profile=dict(driver='GTiff',height=1000,width=1000,count=1,dtype='float32',
        nodata=-9999.0,crs=sar_crs,transform=transform_20m,compress='deflate')
    with rasterio.open(path,'w',**profile) as ds:
        ds.write(np.where(np.isfinite(array),array,-9999.0).astype(np.float32),1)
        ds.set_band_description(1,description)
    return path
ndfi_delta_output=save_f32_map(RASTER_DIR/'Sutlej2025_NDFI_WetMinusDry_Continuous_20m.tif',
                               ndfi_delta_20m,'NDFI_wet_minus_dry')
print('Optical transition:',transition_output)
print('Continuous change:',ndfi_delta_output)


In [ ]:
valid_evidence = valid_change & np.isin(sar_binary_20m,[0,1])
optical_new=(ndfi_transition==1)
sar_water=(sar_binary_20m==1)
new_evidence=np.full((1000,1000),NODATA_CODE,dtype=np.uint8)
new_evidence[valid_evidence & (~optical_new)&(~sar_water)]=0  # neither evidence
new_evidence[valid_evidence & optical_new & sar_water]=1     # corroborated new inundation candidate
new_evidence[valid_evidence & optical_new & (~sar_water)]=2  # optical new water only
new_evidence[valid_evidence & (~optical_new)&sar_water]=3    # SAR wet water; no optical NEW transition
new_evidence_names={
    0:'Neither optical new water nor SAR wet water',
    1:'NDFI new-water transition AND wet SAR water (corroborated candidate)',
    2:'NDFI new-water transition ONLY (not identified by SAR)',
    3:'SAR wet-season water WITHOUT optical new-water transition',
    225:'NoData in SAR or either NDFI period',
}
evidence_areas=pd.DataFrame([dict(code=code,meaning=label,
    pixels=int((new_evidence==code).sum()),area_km2=float((new_evidence==code).sum())*.0004)
    for code,label in new_evidence_names.items()])
display(evidence_areas)
evidence_areas.to_csv(SUPP_TABLE_DIR/'SAR_Wet_and_NDFI_NewInundation_AreaSummary.csv',index=False)
evidence_output=save_u8_map(RASTER_DIR/'Sutlej2025_OpticalNewWater_WetSAR_Evidence4Class_20m.tif',
    new_evidence,'new_inundation_evidence',
    {'1':'Optical dry-to-wet water transition AND wet-season SAR water',
     '2':'Optical transition only','3':'SAR water; NOT optical new-water',
     '0':'Neither','225':'NoData','accuracy':'not independently established'})
# Optional binary subset; 0 includes optical-only and SAR-only disagreements.
strict_candidate=np.full((1000,1000),NODATA_CODE,dtype=np.uint8)
strict_candidate[valid_evidence]=0
strict_candidate[new_evidence==1]=1
strict_output=save_u8_map(RASTER_DIR/'Sutlej2025_OpticalNewWater_AND_WetSAR_StrictCandidate_20m.tif',
    strict_candidate,'optical_newwater_and_wetSAR',
    {'1':'both optical new-water and wet-season SAR water',
     '0':'other valid; includes disagreement','225':'NoData'})
print('New-water evidence:',evidence_output)
print('Strict corroborated subset:',strict_output)


In [ ]:
margin_records=[]
raw_new=valid_change & (d<0) & (w>=0)
for margin,min_delta in NDFI_MARGIN_SCENARIOS:
    eligible=valid_change & (d < -margin) & (w >= margin) & (ndfi_delta_20m >= min_delta)
    overlap=eligible & sar_water & valid_evidence
    margin_records.append(dict(ndfi_margin=margin,min_ndfi_increase=min_delta,
        optical_new_candidate_pixels=int(eligible.sum()),
        optical_new_candidate_km2=float(eligible.sum())*.0004,
        wetSAR_corroborated_pixels=int(overlap.sum()),
        wetSAR_corroborated_km2=float(overlap.sum())*.0004,
        fraction_of_raw_zero_threshold_new_water=float(eligible.sum()/raw_new.sum()) if raw_new.any() else np.nan))
margin_sensitivity=pd.DataFrame(margin_records)
margin_sensitivity.to_csv(SENS_DIR/'NDFI_NewWater_Margin_Sensitivity.csv',index=False)
display(margin_sensitivity)


## 8. Reference-point validation (proxy ground truth)

The `Validation_Points_2025.shp` attribute `Class` defines **0 = non-flood** and **1 = flood**. Sample the two **wet-season binary water classifications separately**, report missing-data exclusions, and repeat on exactly the same valid points for a direct comparison. References are not used to fit or tune either classification. These point-based results evaluate agreement with the labeled reference dataset, whose source and independence must be described in the manuscript.


In [ ]:
references = gpd.read_file(REFERENCE_SHP)
if references.crs is None:
    raise ValueError('Reference points have no CRS. Verify the shapefile .prj file.')
if not references.geometry.geom_type.eq('Point').all():
    raise ValueError('Reference shapefile must contain point geometries only.')
if 'Class' not in references.columns:
    raise KeyError("Reference data needs an integer 'Class' field (0=non-flood, 1=flood).")
if references['Class'].isna().any():
    raise ValueError('Some reference points have missing Class labels.')
reference_labels = pd.to_numeric(references['Class'],errors='raise')
if not reference_labels.isin([0,1]).all():
    raise ValueError('Reference Class must contain only 0 and 1.')
references = references.to_crs(EXPECTED_CRS).reset_index(drop=True)
references['Class'] = reference_labels.to_numpy(dtype=np.int8)
if len(references) != 400:
    warnings.warn(f'Expected 400 reference points; found {len(references)}. Review the source.')
repeated_coordinates = int(references.geometry.duplicated().sum())
if repeated_coordinates:
    warnings.warn(f'{repeated_coordinates} duplicated point coordinates; assess spatial dependence.')

x = references.geometry.x.to_numpy()
y = references.geometry.y.to_numpy()
rows, cols = rowcol(transform_20m, x, y)
rows, cols = np.asarray(rows,dtype=int), np.asarray(cols,dtype=int)
inside = ((rows>=0)&(rows<1000)&(cols>=0)&(cols<1000))

def extract_at_points(raster):
    values = np.full(len(references), 225, dtype=np.uint8)
    values[inside] = raster[rows[inside],cols[inside]]
    return values

sar_point = extract_at_points(sar_binary_20m)
ndfi_point = extract_at_points(ndfi_binary_20m)
truth = references['Class'].to_numpy(dtype=np.uint8)
sar_available = inside & np.isin(sar_point,[0,1])
ndfi_available = inside & np.isin(ndfi_point,[0,1])
shared_available = sar_available & ndfi_available
if not shared_available.any():
    raise ValueError('No reference points have valid values in both wet-season maps.')

samples = pd.DataFrame({
    'point_id': np.arange(1,len(references)+1),
    'easting_m': x, 'northing_m': y,
    'reference_class': truth,
    'SAR_predicted_class': sar_point,
    'NDFI_predicted_class': ndfi_point,
    'inside_study_area': inside,
    'SAR_valid': sar_available,
    'NDFI_valid': ndfi_available,
    'shared_valid': shared_available,
})
samples.to_csv(VALIDATION_DIR/'Reference_Point_Predictions.csv',index=False)

reference_audit = pd.DataFrame([{
    'total_reference_points': len(references),
    'reference_nonflood': int((truth==0).sum()),
    'reference_flood': int((truth==1).sum()),
    'outside_study_area': int((~inside).sum()),
    'duplicate_coordinate_count': repeated_coordinates,
    'SAR_valid_points': int(sar_available.sum()),
    'NDFI_valid_points': int(ndfi_available.sum()),
    'common_valid_points': int(shared_available.sum()),
    'SAR_excluded_points': int((~sar_available).sum()),
    'NDFI_excluded_points': int((~ndfi_available).sum()),
    'common_excluded_points': int((~shared_available).sum()),
}])
reference_audit.to_csv(VALIDATION_DIR/'Reference_Point_Audit.csv',index=False)
display(reference_audit)


In [ ]:
def reference_metrics(label, predicted, available, cohort):
    y_true, y_pred = truth[available], predicted[available]
    if len(y_true)==0:
        raise ValueError(f'No valid samples for {label} ({cohort})')
    matrix = confusion_matrix(y_true,y_pred,labels=[0,1])
    tn, fp, fn, tp = map(int, matrix.ravel())
    out = dict(method=label, cohort=cohort, total_reference=len(truth),
               evaluated_points=len(y_true), excluded_points=len(truth)-len(y_true),
               TN=tn,FP=fp,FN=fn,TP=tp,
               overall_accuracy=accuracy_score(y_true,y_pred),
               precision=precision_score(y_true,y_pred,zero_division=0),
               recall_sensitivity=recall_score(y_true,y_pred,zero_division=0),
               specificity=(tn/(tn+fp) if tn+fp else np.nan),
               F1=f1_score(y_true,y_pred,zero_division=0),
               balanced_accuracy=balanced_accuracy_score(y_true,y_pred),
               cohens_kappa=cohen_kappa_score(y_true,y_pred))
    return out,matrix

validation_rows = []
matrices = {}
for name, predictions, available in [
    ('SAR',sar_point,sar_available), ('NDFI',ndfi_point,ndfi_available)]:
    for cohort, mask in [('map_valid',available),('common_valid',shared_available)]:
        metrics, matrix = reference_metrics(name,predictions,mask,cohort)
        validation_rows.append(metrics)
        matrices[name,cohort] = matrix
        pd.DataFrame(matrix,index=['Reference 0','Reference 1'],
                     columns=['Predicted 0','Predicted 1']).to_csv(
                         VALIDATION_DIR/f'{name}_Confusion_Matrix_{cohort}.csv')

validation_metrics = pd.DataFrame(validation_rows)
validation_metrics.to_csv(VALIDATION_DIR/'Reference_Validation_Metrics_All.csv',index=False)
main_validation = validation_metrics.query("cohort == 'common_valid'").copy()
main_validation.to_csv(MAIN_TABLE_DIR/'Table_02_Reference_Validation.csv',index=False)
display(main_validation.round(3))


## 9. Manuscript figures and results tables

Figures are 600-DPI TIFFs with muted, colorblind-aware categorical palettes and fixed physical dimensions. Scientific GeoTIFF values and resolutions are unchanged. The main selection is limited to seasonal RGB, wet-sensor agreement, corroborated optical change, and validation; diagnostic figures are supplementary.


In [ ]:
ACADEMIC = {
    'navy':'#274862','teal':'#257E77','ochre':'#BC9548',
    'rust':'#B65D53','gray':'#E7E9E9','ink':'#27333A',
    'soft_blue':'#C4D9E1',
}
FUSION_PALETTE = {0:ACADEMIC['gray'],1:ACADEMIC['navy'],
                  2:ACADEMIC['ochre'],3:ACADEMIC['teal']}
EVIDENCE_PALETTE = {0:ACADEMIC['gray'],1:ACADEMIC['navy'],
                    2:ACADEMIC['ochre'],3:ACADEMIC['rust']}
TRANSITION_PALETTE = {0:ACADEMIC['gray'],1:ACADEMIC['teal'],
                      2:ACADEMIC['navy'],3:ACADEMIC['ochre']}
BINARY_PALETTE = {0:ACADEMIC['gray'],1:ACADEMIC['navy']}
MAP_EXTENT_KM = (transform_20m.c/1000,
                 (transform_20m.c+1000*transform_20m.a)/1000,
                 (transform_20m.f+1000*transform_20m.e)/1000,
                 transform_20m.f/1000)

def figure_inches(width_mm=FIG_WIDTH_MM,height_mm=105):
    return width_mm/25.4,height_mm/25.4

def save_figure(fig,path):
    fig.savefig(path,format='tiff',dpi=FIG_DPI,facecolor='white',
                pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig)
    with Image.open(path) as im:
        dpi = im.info.get('dpi',(0,0))
        if abs(dpi[0]-FIG_DPI)>1 or abs(dpi[1]-FIG_DPI)>1:
            raise AssertionError(f'Wrong TIFF resolution: {path}, {dpi}')
    print('Figure:',path.relative_to(PROJECT_DIR))

def map_axes(ax,title):
    ax.set_title(title,loc='left',fontsize=10,weight='bold',pad=7)
    ax.set(xlabel='UTM easting (km)',ylabel='UTM northing (km)')
    ax.set_xlim(MAP_EXTENT_KM[:2]);ax.set_ylim(MAP_EXTENT_KM[2:])
    ax.set_aspect('equal',adjustable='box')
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    for side in ax.spines.values(): side.set_visible(True)
    # Scale bar and north marker, derived from the 20-km coordinate extent.
    x0,y0=MAP_EXTENT_KM[0]+1.0,MAP_EXTENT_KM[2]+1.0
    ax.add_patch(Rectangle((x0-.25,y0-.35),5.5,1.3,facecolor='white',alpha=.85,edgecolor='none'))
    ax.plot([x0,x0+5],[y0,y0],color=ACADEMIC['ink'],lw=2.1)
    ax.text(x0+2.5,y0+.18,'5 km',ha='center',va='bottom',fontsize=8)
    ax.annotate('N',xy=(.94,.94),xytext=(.94,.81),xycoords='axes fraction',
                textcoords='axes fraction',ha='center',fontsize=9,weight='bold',
                arrowprops=dict(arrowstyle='-|>',lw=1.2,color=ACADEMIC['ink']))

def classified(ax,array,palette):
    rgba=np.ones(array.shape+(4,),dtype=np.float32)
    from matplotlib.colors import to_rgba
    for key,color in palette.items(): rgba[array==key]=to_rgba(color)
    rgba[array==225]=to_rgba('#FFFFFF')
    ax.imshow(rgba,extent=MAP_EXTENT_KM,origin='upper',interpolation='nearest')

def legend(fig,palette,labels,columns=3,y=.015):
    handles=[Patch(facecolor=palette[i],edgecolor='#666',linewidth=.3,label=label)
             for i,label in labels]
    fig.legend(handles=handles,ncol=columns,frameon=False,
               loc='lower center',bbox_to_anchor=(.5,y),fontsize=8)

def read_rgb(path):
    with rasterio.open(path) as ds:
        if ds.count<3 or ds.crs!=sar_crs or ds.width<2000 or ds.height<2000:
            raise ValueError(f'Invalid RGB raster: {path.name}')
        if not np.allclose(tuple(ds.transform)[:6],tuple(transform_10m)[:6],rtol=0,atol=1e-8):
            raise ValueError(f'RGB grid not aligned: {path.name}')
        arr=ds.read([1,2,3],window=rasterio.windows.Window(0,0,2000,2000),masked=True)
        band_masks=np.ma.getmaskarray(arr)
        values=arr.astype(np.float32).filled(np.nan)
        if np.nanpercentile(values,99)>2: values /= 10000.0
        result=np.clip(values/.3,0,1)**(1/1.1)
        result=np.moveaxis(np.nan_to_num(result,nan=1.0),0,-1)
        result[band_masks.any(axis=0)]=1.0
        return result


In [ ]:
# Main Figure 1: measured dry/wet surface-reflectance RGB.
fig,axes=plt.subplots(1,2,figsize=figure_inches(180,105))
for ax,img,title in zip(axes,(read_rgb(RGB_DRY),read_rgb(RGB_WET)),
    ('A  Dry season (May–Jun 2025)','B  Wet season (Sep 2025)')):
    ax.imshow(img,extent=MAP_EXTENT_KM,origin='upper',interpolation='nearest')
    map_axes(ax,title)
fig.subplots_adjust(left=.09,right=.99,bottom=.13,top=.91,wspace=.34)
save_figure(fig,MAIN_FIG_DIR/'Fig_01_Seasonal_Sentinel2_RGB.tiff')

# Main Figure 2: inter-method wet-season water agreement (not validation).
fig,ax=plt.subplots(figsize=figure_inches(180,145))
classified(ax,fusion_4class,FUSION_PALETTE)
map_axes(ax,'Wet-season SAR–NDFI water agreement')
legend(fig,FUSION_PALETTE,[(0,'Neither'),(1,'Both water'),
                           (2,'SAR only'),(3,'NDFI only')],columns=4,y=.02)
fig.subplots_adjust(left=.14,right=.93,bottom=.16,top=.94)
save_figure(fig,MAIN_FIG_DIR/'Fig_02_WetWater_Intermethod_Agreement.tiff')

# Main Figure 3: optical transition and cross-sensor corroboration.
fig,axes=plt.subplots(1,2,figsize=figure_inches(180,115))
classified(axes[0],ndfi_transition,TRANSITION_PALETTE)
classified(axes[1],new_evidence,EVIDENCE_PALETTE)
map_axes(axes[0],'A  NDFI dry-to-wet transitions')
map_axes(axes[1],'B  New-water evidence and SAR')
fig.legend(handles=[Patch(facecolor=TRANSITION_PALETTE[k],edgecolor='#666',lw=.3,label=txt)
    for k,txt in [(0,'Stable land'),(1,'New water'),(2,'Persistent water'),(3,'Receded water')]],
    loc='lower center',bbox_to_anchor=(.5,.075),ncol=4,fontsize=7.5,frameon=False)
fig.legend(handles=[Patch(facecolor=EVIDENCE_PALETTE[k],edgecolor='#666',lw=.3,label=txt)
    for k,txt in [(0,'Neither'),(1,'Both criteria'),(2,'Optical new only'),(3,'SAR wet only')]],
    loc='lower center',bbox_to_anchor=(.5,.01),ncol=4,fontsize=7.5,frameon=False)
fig.subplots_adjust(left=.09,right=.99,bottom=.25,top=.93,wspace=.35)
save_figure(fig,MAIN_FIG_DIR/'Fig_03_Candidate_New_Inundation.tiff')


In [ ]:
# Main Figure 4: confusion matrices, evaluated on the SAME valid reference points.
fig,axes=plt.subplots(1,2,figsize=figure_inches(180,94))
for ax,method in zip(axes,('SAR','NDFI')):
    cm=matrices[method,'common_valid']
    ax.imshow(cm,cmap=ListedColormap(['#F2F4F3','#BDD0D1','#3B777A']))
    for (r,c),number in np.ndenumerate(cm):
        ax.text(c,r,f'{number:,}',ha='center',va='center',fontsize=12,
                fontweight='bold',color=ACADEMIC['ink'])
    ax.set_xticks([0,1],['Non-flood','Flood'])
    ax.set_yticks([0,1],['Non-flood','Flood'])
    ax.set_xlabel('Mapped class')
    ax.set_ylabel('Reference class')
    row=main_validation.loc[main_validation.method==method].iloc[0]
    ax.set_title(f'{method}  |  OA {row.overall_accuracy:.3f}  |  F1 {row.F1:.3f}',weight='bold')
fig.suptitle(f'Reference-point assessment (n={int(shared_available.sum())} jointly valid points)',fontsize=11)
fig.subplots_adjust(left=.13,right=.98,bottom=.17,top=.80,wspace=.58)
save_figure(fig,MAIN_FIG_DIR/'Fig_04_Reference_Validation_Confusion_Matrices.tiff')

# Concise figure-ready summary table, also provided as editable CSV.
main_summary = pd.DataFrame([
    ('Wet SAR water',int((sar_binary_20m==1).sum())*.0004,'km²'),
    ('Wet NDFI water',int((ndfi_binary_20m==1).sum())*.0004,'km²'),
    ('Optical newly appearing water',int((ndfi_transition==1).sum())*.0004,'km²'),
    ('Optical new water ∩ wet SAR',int((new_evidence==1).sum())*.0004,'km²'),
    ('Wet SAR / NDFI observed agreement',observed_agreement*100,'%'),
    ('Wet SAR / NDFI Cohen κ',kappa,'unitless'),
],columns=['Measure','Value','Unit'])
main_summary.to_csv(MAIN_TABLE_DIR/'Table_01_Area_and_Agreement.csv',index=False)
validation_metrics.to_csv(SUPP_TABLE_DIR/'Table_S04_Reference_Validation_All_Cohorts.csv',index=False)

# TIFF table previews accompany editable CSV tables.
def save_table_tiff(dataframe, title, path, height_mm=75):
    fig,ax=plt.subplots(figsize=figure_inches(180,height_mm))
    ax.axis('off')
    grid=ax.table(cellText=dataframe.values,colLabels=dataframe.columns,
                  cellLoc='left',colLoc='left',loc='center',bbox=[.02,.08,.96,.78])
    grid.auto_set_font_size(False);grid.set_fontsize(8.2)
    for (row,col),cell in grid.get_celld().items():
        cell.set_edgecolor('#D7DEDF')
        cell.set_linewidth(.35)
        cell.set_facecolor('#DCE7EA' if row==0 else ('white' if row%2 else '#F3F6F6'))
        if row==0:cell.set_text_props(weight='bold',color=ACADEMIC['ink'])
    fig.suptitle(title,x=.03,y=.96,ha='left',fontsize=11,weight='bold')
    save_figure(fig,path)

area_table=main_summary.copy()
area_table['Value']=area_table['Value'].map(lambda val:f'{val:.3f}')
save_table_tiff(area_table,'Table 1  |  Mapped extent and inter-method agreement',
                MAIN_TABLE_DIR/'Table_01_Area_and_Agreement.tiff')
validation_table=pd.DataFrame({
    'Method':main_validation.method.to_numpy(),
    'n':main_validation.evaluated_points.astype(str).to_numpy(),
    'OA':main_validation.overall_accuracy.map(lambda v:f'{v:.3f}').to_numpy(),
    'Prec.':main_validation.precision.map(lambda v:f'{v:.3f}').to_numpy(),
    'Sens.':main_validation.recall_sensitivity.map(lambda v:f'{v:.3f}').to_numpy(),
    'Spec.':main_validation.specificity.map(lambda v:f'{v:.3f}').to_numpy(),
    'F1':main_validation.F1.map(lambda v:f'{v:.3f}').to_numpy(),
    'κ':main_validation.cohens_kappa.map(lambda v:f'{v:.3f}').to_numpy(),
})
save_table_tiff(validation_table,'Table 2  |  Reference-point assessment (common valid points)',
                MAIN_TABLE_DIR/'Table_02_Reference_Validation.tiff',height_mm=60)


In [ ]:
# Supplementary figures: diagnostics that help explain, but do not select, settings.
from scipy.stats import norm
xx=np.linspace(np.percentile(wet_values,.1),np.percentile(wet_values,99.9),800)
pdf_low=priors[water_component]*norm.pdf(xx,means[water_component],sigmas[water_component])
pdf_high=priors[background_component]*norm.pdf(xx,means[background_component],sigmas[background_component])
fig,ax=plt.subplots(figsize=figure_inches(150,85))
ax.hist(training.ravel(),bins=110,density=True,color=ACADEMIC['soft_blue'],label='Observed wet VV')
ax.plot(xx,pdf_low,color=ACADEMIC['navy'],label='Lower-backscatter component')
ax.plot(xx,pdf_high,color=ACADEMIC['ochre'],label='Higher-backscatter component')
ax.plot(xx,pdf_low+pdf_high,color=ACADEMIC['ink'],linestyle='--',label='Mixture')
ax.set(xlabel='Wet VV backscatter (dB)',ylabel='Density')
ax.legend(frameon=False,fontsize=8)
fig.subplots_adjust(left=.14,right=.98,bottom=.19,top=.95)
save_figure(fig,SUPP_FIG_DIR/'Fig_S01_GMM_WetVV_Density.tiff')

if RUN_SAR_SENSITIVITY and not sensitivity.empty:
    subset=sensitivity[(sensitivity.n_segments_requested==SELECTED['n_segments']) &
                       (sensitivity.compactness==SELECTED['compactness'])].sort_values('graphcut_lambda')
    fig,ax=plt.subplots(figsize=figure_inches(150,85))
    ax.plot(subset.graphcut_lambda,subset.water_area_km2_10m,'o-',color=ACADEMIC['navy'])
    ax.set(xlabel='Graph-cut smoothness λ',ylabel='Candidate water area at 10 m (km²)')
    ax.grid(alpha=.16)
    fig.subplots_adjust(left=.16,right=.98,bottom=.18,top=.95)
    save_figure(fig,SUPP_FIG_DIR/'Fig_S02_GraphCut_Sensitivity.tiff')

fig,ax=plt.subplots(figsize=figure_inches(150,90))
xpos=np.arange(len(margin_sensitivity))
ax.plot(xpos,margin_sensitivity.optical_new_candidate_km2,'o-',
        color=ACADEMIC['teal'],label='Optical new-water area')
ax.plot(xpos,margin_sensitivity.wetSAR_corroborated_km2,'s--',
        color=ACADEMIC['navy'],label='SAR-corroborated area')
ax.set_xticks(xpos,[f'{m:.2f} / {delta:.2f}' for m,delta in NDFI_MARGIN_SCENARIOS])
ax.set(xlabel='NDFI margin / minimum increase',ylabel='Candidate area (km²)')
ax.legend(frameon=False,fontsize=8);ax.grid(alpha=.16)
fig.subplots_adjust(left=.16,right=.98,bottom=.22,top=.95)
save_figure(fig,SUPP_FIG_DIR/'Fig_S03_NDFI_Margin_Sensitivity.tiff')


## 9A. Publication selections and plotting defaults

Four main figures convey seasonal context, cross-sensor agreement, evidence for candidate new inundation, and the image-interpreted reference comparison. Five concise supplementary diagnostics cover the GMM fit, spatial-block sensitivity, disagreement geography, NDFI margins, and SAR parameter stability. The original figure exports remain available for audit; **use the curated set rather than both sets in the submission**.

Tables are exported as editable CSV, not flattened images. Each figure is exported as a 600-DPI LZW-compressed TIFF only; no PDF is created. The TIFF describes a printed figure and is not a replacement for the georeferenced GeoTIFF datasets. Map coordinates are UTM kilometres (EPSG:32642); mapped areas are descriptive quantities, not unbiased flood-area estimates.


In [ ]:
from matplotlib.colors import Normalize
from matplotlib.ticker import PercentFormatter

PUB_DIR = OUTPUT_ROOT / '07_Publication_Ready'
PUB_MAIN = PUB_DIR / 'Manuscript'
PUB_SUPP = PUB_DIR / 'Supplementary'
PUB_TABLES = PUB_DIR / 'Tables'
for folder in (PUB_MAIN, PUB_SUPP, PUB_TABLES):
    folder.mkdir(parents=True, exist_ok=True)

# Physical dimensions are fixed; typography and color are consistent across panels.
PUB_STYLE = {
    'font.family': 'DejaVu Sans', 'font.size': 8.4,
    'axes.titlesize': 9.6, 'axes.labelsize': 8.4,
    'xtick.labelsize': 7.4, 'ytick.labelsize': 7.4,
    'legend.fontsize': 7.5,
    'savefig.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
}

def export_figure(fig, folder, stem):
    """Export one LZW TIFF at 600 DPI; no vector/PDF companion."""
    target = folder / f'{stem}.tiff'
    fig.savefig(target, format='tiff', dpi=FIG_DPI, facecolor='white',
                pil_kwargs={'compression': 'tiff_lzw'})
    plt.close(fig)
    # A prior v2.0 run may have left this auto-generated companion PDF.
    old_companion = folder / f'{stem}.pdf'
    if old_companion.exists():
        old_companion.unlink()
    with Image.open(target) as im:
        dpi = im.info.get('dpi', (0, 0))
        if abs(dpi[0] - FIG_DPI) > 1 or abs(dpi[1] - FIG_DPI) > 1:
            raise AssertionError(f'Incorrect figure resolution: {target}: {dpi}')
    print('Exported:', target.relative_to(PROJECT_DIR))

def paper_map(ax, array, palette, title):
    classified(ax, array, palette)
    map_axes(ax, title)
    ax.tick_params(length=2.5)

caption_records = []
def register(name, destination, description):
    caption_records.append({'figure': name, 'placement': destination,
                            'caption_draft': description})


## 9B. Spatial uncertainty and disagreement diagnostics

Spatial block bootstrap intervals are **approximate confidence intervals for inter-method agreement**, not confidence intervals for mapped flood extent or reference-based accuracy. Compare 0.5, 1 and 2 km blocks to disclose dependence on an arbitrary spatial resampling scale. Sampling is repeatable using fixed random seeds, and the original 1 km analysis remains unchanged.

A separate 1 km grid summarizes where SAR and NDFI differ, using only cells observed by both sensors; blocks with zero joint coverage remain missing. No reference labels are used in either diagnostic.


In [ ]:
def block_confusion_counts(sar_mask, optical_mask, block_px):
    h, w = sar_mask.shape
    if h % block_px or w % block_px:
        raise ValueError('Block size must divide the 20 m raster dimensions.')
    valid = np.isin(sar_mask, [0, 1]) & np.isin(optical_mask, [0, 1])
    counts = []
    for a in (0, 1):
        for b in (0, 1):
            matching = valid & (sar_mask == a) & (optical_mask == b)
            tile = matching.reshape(h//block_px, block_px, w//block_px, block_px)
            counts.append(tile.sum(axis=(1, 3)).ravel())
    return np.stack(counts, axis=1)

def agreement_from_counts(flat_counts):
    m = np.asarray(flat_counts, dtype=float).reshape(2, 2)
    n = m.sum()
    if n == 0:
        return np.nan, np.nan
    expected = (m.sum(axis=0) * m.sum(axis=1)).sum() / n**2
    kap = (np.trace(m)/n - expected) / (1 - expected) if expected < 1 else np.nan
    union = m[1, 1] + m[1, 0] + m[0, 1]
    iou = m[1, 1]/union if union > 0 else np.nan
    return kap, iou

block_records = []
for width in (25, 50, 100):  # 500, 1000, 2000 m on the aligned 20 m grid
    histograms = block_confusion_counts(sar_binary_20m, ndfi_binary_20m, width)
    rng_blocks = np.random.default_rng(SEED + 900 + width)
    draws = rng_blocks.integers(0, len(histograms),
                                size=(BOOTSTRAP_REPLICATES, len(histograms)))
    totals = histograms[draws].sum(axis=1)
    values = np.array([agreement_from_counts(c) for c in totals])
    if not np.isfinite(values).all():
        raise RuntimeError('Undefined bootstrap statistics; inspect the joint-valid raster.')
    q_kappa = np.quantile(values[:, 0], [.025, .975])
    q_iou = np.quantile(values[:, 1], [.025, .975])
    block_records.append(dict(block_size_m=20*width,
        sampled_blocks=len(histograms), replicates=len(values),
        kappa=kappa, kappa_lo=q_kappa[0], kappa_hi=q_kappa[1],
        water_IoU=water_iou, IoU_lo=q_iou[0], IoU_hi=q_iou[1]))
block_results = pd.DataFrame(block_records)
block_results.to_csv(PUB_TABLES/'Table_S01_Block_Bootstrap_Sensitivity.csv', index=False)

# One-kilometre spatial grid of per-tile disagreement, not classification error.
tile_width = 50
tile_valid = valid_pair.reshape(20, tile_width, 20, tile_width).sum(axis=(1, 3))
different = valid_pair & (sar_binary_20m != ndfi_binary_20m)
tile_different = different.reshape(20, tile_width, 20, tile_width).sum(axis=(1, 3))
tile_pct = np.divide(100*tile_different, tile_valid,
                     out=np.full((20, 20), np.nan, dtype=float), where=tile_valid > 0)
tile_rows = []
for row in range(20):
    for col in range(20):
        tile_rows.append(dict(row_north_to_south=row, column_west_to_east=col,
            valid_joint_pixels=int(tile_valid[row, col]),
            discordant_pixels=int(tile_different[row, col]),
            discordance_pct=float(tile_pct[row, col])))
pd.DataFrame(tile_rows).to_csv(PUB_TABLES/'Table_S02_Spatial_Disagreement_1km.csv', index=False)

assert np.isclose(kappa, agreement_from_counts(confusion.ravel())[0], atol=1e-12)
assert int(tile_valid.sum()) == n_compared
print(block_results.round(3).to_string(index=False))


## 9C. Selected manuscript figures and tables

Use these as four candidate manuscript figures. Their labels deliberately distinguish mapped water from confirmed inundation and image-interpreted reference agreement from field validation. Captions must be checked against the revised manuscript before submission.

For Figure 2, class labels are inside the summary panel rather than outside its y-axis; this prevents them from covering map pixels.


In [ ]:
from matplotlib.patches import Patch

with plt.rc_context(PUB_STYLE):
    # Figure 1. The same area and coordinate system in both seasonal RGB panels.
    fig, axes = plt.subplots(1, 2, figsize=figure_inches(180, 103))
    for ax, source, label in zip(axes, (RGB_DRY, RGB_WET),
                                ('A  May–June 2025', 'B  September 2025')):
        ax.imshow(read_rgb(source), extent=MAP_EXTENT_KM, origin='upper')
        map_axes(ax, label)
    fig.subplots_adjust(left=.085, right=.985, bottom=.13, top=.91, wspace=.31)
    export_figure(fig, PUB_MAIN, 'Fig01_Seasonal_Optical_Context')
    register('Fig. 1', 'Manuscript',
        'Sentinel-2 surface-reflectance RGB composites of the 20 × 20 km Sutlej window '
        'in the dry and wet observation periods. Common UTM extent; white denotes invalid RGB data.')

    # Figure 2. Labels sit entirely inside panel B (no intrusion into the map).
    fig = plt.figure(figsize=figure_inches(180, 111))
    ax_map = fig.add_axes([.095, .16, .46, .75])
    ax_bar = fig.add_axes([.625, .26, .335, .55])
    paper_map(ax_map, fusion_4class, FUSION_PALETTE,
              'A  Wet-season water agreement')
    plot_codes = [0, 1, 2, 3]
    plot_labels = ['Neither', 'Both water', 'SAR only', 'NDFI only']
    amounts = np.array([int((fusion_4class == c).sum()) for c in plot_codes])
    pct = 100 * amounts / n_compared
    # A label is displayed above each bar instead of as an external y-axis tick.
    # This avoids collisions with the mapped area even after journal resizing.
    rows = np.arange(len(plot_codes))[::-1]
    for y, c, name, share in zip(rows, plot_codes, plot_labels, pct):
        ax_bar.barh(y, share, color=FUSION_PALETTE[c], height=.35)
        ax_bar.text(0, y + .26, name, ha='left', va='bottom', fontsize=7.4)
        ax_bar.text(min(share + 1.3, 103), y, f'{share:.1f}%',
                    ha='left', va='center', fontsize=7.3)
    ax_bar.set_yticks([])
    ax_bar.set_ylim(-.50, 3.65)
    ax_bar.set_xlim(0, 108)
    ax_bar.set_xticks([0, 20, 40, 60, 80, 100])
    ax_bar.set_xlabel('% of jointly valid cells')
    ax_bar.set_title('B  Class shares', loc='left', weight='bold', pad=9)
    ax_bar.spines['left'].set_visible(False)
    ax_bar.tick_params(axis='y', length=0)
    fig.text(.625, .10, f'n = {n_compared:,} jointly valid 20 m cells',
             fontsize=7.4)
    # Guard against layout regressions: panel-B text must not cross into panel A.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    map_right = ax_map.get_window_extent(renderer).x1
    for artist in ax_bar.texts:
        bounds = artist.get_window_extent(renderer)
        if bounds.x0 < map_right + 2:
            raise AssertionError('Figure 2 label overlaps the map panel.')
        if bounds.x1 > fig.bbox.x1 - 2:
            raise AssertionError('Figure 2 label extends beyond the figure.')
    export_figure(fig, PUB_MAIN, 'Fig02_SAR_NDFI_WetAgreement')
    register('Fig. 2', 'Manuscript',
        'Wet-season Sentinel-1/2 candidate-water agreement (A) and shares of jointly '
        'valid 20 m cells (B). Agreement is not independent classification accuracy.')

    # Figure 3. Comparison uses shared 20 m coordinate geometry; distinct legends.
    fig, axes = plt.subplots(1, 2, figsize=figure_inches(180, 123))
    paper_map(axes[0], ndfi_transition, TRANSITION_PALETTE,
              'A  Optical dry-to-wet transition')
    paper_map(axes[1], new_evidence, EVIDENCE_PALETTE,
              'B  Optical change × wet SAR')
    first = [(0, 'Stable non-water'), (1, 'New-water candidate'),
             (2, 'Water both periods'), (3, 'Water disappearance')]
    second = [(0, 'Neither'), (1, 'Both criteria'),
              (2, 'Optical only'), (3, 'Wet SAR only')]
    for y, selected, pal in ((.087, first, TRANSITION_PALETTE),
                             (.035, second, EVIDENCE_PALETTE)):
        fig.legend([Patch(facecolor=pal[c], edgecolor='#666', linewidth=.3, label=text)
                    for c, text in selected],
                   [text for _, text in selected], loc='lower center', ncol=4,
                   bbox_to_anchor=(.5, y), frameon=False, fontsize=7.2)
    fig.subplots_adjust(left=.083, right=.985, bottom=.25, top=.93, wspace=.31)
    export_figure(fig, PUB_MAIN, 'Fig03_CandidateNewWater_Evidence')
    register('Fig. 3', 'Manuscript',
        'Optical dry-to-wet water transitions and areas also classified as wet-season SAR '
        'water. Concurrence supports a candidate-inundation interpretation; neither map confirms flood origin.')

    # Figure 4. Balanced optical-image reference; NOT an independent field-accuracy sample.
    fig, axes = plt.subplots(1, 2, figsize=figure_inches(180, 93))
    for ax, method in zip(axes, ('SAR', 'NDFI')):
        cm = matrices[method, 'common_valid']
        denominators = cm.sum(axis=1, keepdims=True)
        proportions = np.divide(cm, denominators, out=np.zeros_like(cm, dtype=float),
                                where=denominators > 0)
        ax.imshow(proportions, cmap='Blues', vmin=0, vmax=1)
        for (r, c), count in np.ndenumerate(cm):
            fraction = proportions[r, c]
            ax.text(c, r, f'{count:,}\n({100*fraction:.1f}%)',
                    ha='center', va='center', color=('white' if fraction > .6 else '#222'),
                    fontsize=8, weight='bold')
        ax.set_xticks([0, 1], ['Non-water', 'Water'])
        ax.set_yticks([0, 1], ['Non-water', 'Water'])
        ax.set_xlabel('Mapped wet-season class')
        ax.set_ylabel('Image-interpreted class')
        agreement = np.trace(cm) / cm.sum()
        ax.set_title(f'{method}  |  sample agreement {100*agreement:.2f}%',
                     loc='left', weight='bold')
    fig.text(.5, .96,
        f'Sentinel-2 RGB-interpreted sample (n = {int(shared_available.sum())}); '
        'not field validation', ha='center', fontsize=9)
    fig.subplots_adjust(left=.11, right=.975, bottom=.16, top=.81, wspace=.48)
    export_figure(fig, PUB_MAIN, 'Fig04_ImageInterpretedReference_Agreement')
    register('Fig. 4', 'Manuscript',
        'Confusion counts and class-wise sample agreement relative to purposively '
        'balanced, Sentinel-2 RGB-interpreted labels. The labels are not ground observations '
        'and do not support design-based area-adjusted accuracy.')

# Concise, editable manuscript tables; formats are suitable for journal typesetting.
main_tab1 = pd.DataFrame([
    ['Wet-season SAR candidate water', agreement_metrics['SAR_water_area_whole_grid_km2'], 'km²'],
    ['Wet-season NDFI candidate water', agreement_metrics['wet_NDFI_water_area_whole_grid_km2'], 'km²'],
    ['Optical newly appearing water', int((ndfi_transition == 1).sum())*.0004, 'km²'],
    ['Optical new water also mapped by wet SAR', int((new_evidence == 1).sum())*.0004, 'km²'],
    ['Wet SAR–NDFI observed agreement', 100*observed_agreement, '%'],
    ['Wet SAR–NDFI Cohen κ', kappa, 'unitless'],
    ['Water-class intersection over union', water_iou, 'unitless'],
], columns=['Measure', 'Value', 'Unit'])
main_tab1.to_csv(PUB_TABLES/'Table01_WaterExtent_Agreement.csv', index=False)
cols = ['method', 'evaluated_points', 'TN', 'FP', 'FN', 'TP', 'overall_accuracy',
        'precision', 'recall_sensitivity', 'specificity', 'F1', 'cohens_kappa']
main_validation[cols].to_csv(PUB_TABLES/'Table02_ImageInterpreted_Reference.csv', index=False)


## 9D. Supplementary figures: model fit, uncertainty and sensitivity

The supplementary set is deliberately small. Each panel addresses a specific methodological question without treating a comparator, bootstrap result or posterior mixture probability as ground truth. The SAR parameter figure is omitted if the original sensitivity run was disabled.


In [ ]:
from scipy.stats import norm

with plt.rc_context(PUB_STYLE):
    # S1: Model fit and posterior ambiguity (not a calibrated water probability).
    axis_vv = np.linspace(np.percentile(wet_values, .2),
                          np.percentile(wet_values, 99.8), 500)
    post_curve = gmm.predict_proba(axis_vv.reshape(-1, 1))[:, water_component]
    fig, axes = plt.subplots(1, 2, figsize=figure_inches(180, 82))
    low = priors[water_component] * norm.pdf(axis_vv,
            means[water_component], sigmas[water_component])
    high = priors[background_component] * norm.pdf(axis_vv,
            means[background_component], sigmas[background_component])
    axes[0].hist(training.ravel(), bins=95, density=True, alpha=.66,
                 color=ACADEMIC['soft_blue'], label='VV subsample')
    axes[0].plot(axis_vv, low, color=ACADEMIC['navy'], label='Lower VV')
    axes[0].plot(axis_vv, high, color=ACADEMIC['ochre'], label='Higher VV')
    axes[0].plot(axis_vv, low+high, color=ACADEMIC['ink'], ls='--', label='Mixture')
    axes[0].set(xlabel='Wet VV backscatter (dB)', ylabel='Density')
    axes[0].set_title('A  VV mixture fit', loc='left', weight='bold')
    axes[0].legend(frameon=False, fontsize=6.8)
    axes[1].plot(axis_vv, post_curve, color=ACADEMIC['navy'], lw=1.8)
    axes[1].axhspan(.25, .75, color='#EFF2F1', zorder=0)
    axes[1].axhline(.5, color=ACADEMIC['rust'], ls='--', lw=.9)
    axes[1].set(xlabel='Wet VV backscatter (dB)',
                ylabel='Lower-VV component posterior', ylim=(0, 1))
    axes[1].set_title('B  Mixture ambiguity', loc='left', weight='bold')
    fig.subplots_adjust(left=.095, right=.98, bottom=.19, top=.9, wspace=.36)
    export_figure(fig, PUB_SUPP, 'FigS01_GMM_Fit_And_Ambiguity')
    register('Fig. S1', 'Supplementary',
        'Fitted Gaussian mixture in wet VV dB and lower-backscatter component posterior. '
        'Posterior ambiguity is model-based, not a calibrated probability of flooding.')

    # S2: Effect of spatial block size on 95% inter-method intervals.
    fig, axes = plt.subplots(1, 2, figsize=figure_inches(180, 80))
    for ax, key, lo, hi, title in [
        (axes[0], 'kappa', 'kappa_lo', 'kappa_hi', 'A  Cohen κ'),
        (axes[1], 'water_IoU', 'IoU_lo', 'IoU_hi', 'B  Water IoU')]:
        center = block_results[key].to_numpy()
        low = block_results[lo].to_numpy()
        upper = block_results[hi].to_numpy()
        x = block_results.block_size_m.to_numpy() / 1000
        ax.vlines(x, low, upper, color=ACADEMIC['navy'], lw=1.3)
        ax.scatter(x, center, color=ACADEMIC['navy'], s=23, zorder=3)
        for xi, lower, upper_bound in zip(x, low, upper):
            ax.hlines([lower, upper_bound], xi-.04, xi+.04,
                      color=ACADEMIC['navy'], lw=1.2)
        ax.axhline(float(center[0]), color=ACADEMIC['rust'], lw=.85, ls=':')
        ax.set(xlabel='Square block width (km)', ylabel='Statistic')
        ax.set_xticks(x)
        ax.set_title(title, loc='left', weight='bold')
        ax.grid(axis='y', color='#DDDDDD', lw=.5)
    fig.subplots_adjust(left=.095, right=.98, bottom=.21, top=.87, wspace=.35)
    export_figure(fig, PUB_SUPP, 'FigS02_SpatialBlock_Bootstrap')
    register('Fig. S2', 'Supplementary',
        'Sensitivity of approximate inter-method agreement intervals to 0.5, 1 and 2 km '
        'spatial bootstrap blocks. Intervals quantify agreement uncertainty, not ground-truth accuracy.')

    # S3: Geographical distribution of disagreement on a fixed 1-km assessment grid.
    fig, ax = plt.subplots(figsize=figure_inches(150, 120))
    im = ax.imshow(tile_pct, extent=MAP_EXTENT_KM, origin='upper',
                   cmap='magma_r', vmin=0, vmax=max(10., float(np.nanmax(tile_pct))))
    map_axes(ax, 'Sensor disagreement within 1 km blocks')
    cb = fig.colorbar(im, ax=ax, shrink=.80, pad=.045)
    cb.set_label('Disagreement (% of jointly valid cells)')
    fig.subplots_adjust(left=.13, right=.93, bottom=.14, top=.91)
    export_figure(fig, PUB_SUPP, 'FigS03_Spatial_Disagreement_1km')
    register('Fig. S3', 'Supplementary',
        'Local percentage of mutually valid cells classified differently by SAR and NDFI '
        'in 1 × 1 km grid cells. Disagreement is not synonymous with error.')

    # S4: Optical decision-margin sensitivity with numbers tied to the exported table.
    fig, ax = plt.subplots(figsize=figure_inches(160, 81))
    xx = np.arange(len(margin_sensitivity))
    for key, style, tone, title in [
        ('optical_new_candidate_km2', 'o-', ACADEMIC['teal'], 'Optical new water'),
        ('wetSAR_corroborated_km2', 's--', ACADEMIC['navy'], 'Also wet SAR')]:
        ax.plot(xx, margin_sensitivity[key], style, color=tone, label=title, lw=1.8)
    ax.set_xticks(xx, [f'{m:.2f}, {delta:.2f}' for m, delta in NDFI_MARGIN_SCENARIOS])
    ax.set(xlabel='Dry/wet margin, minimum NDFI increase',
           ylabel='Candidate area (km²)')
    ax.grid(axis='y', color='#DDDDDD', lw=.5)
    ax.legend(frameon=False)
    fig.subplots_adjust(left=.13, right=.98, bottom=.24, top=.90)
    export_figure(fig, PUB_SUPP, 'FigS04_NDFI_Margin_Sensitivity')
    register('Fig. S4', 'Supplementary',
        'Change in candidate newly appearing optical water and SAR-corroborated area '
        'under stricter NDFI decision margins. Area sensitivity does not measure accuracy.')

    # S5: Parameter stability relative to the unchanged baseline, when computed.
    if RUN_SAR_SENSITIVITY and not sensitivity.empty:
        fig, ax = plt.subplots(figsize=figure_inches(160, 81))
        order = sensitivity.sort_values('water_IoU_vs_baseline_10m').reset_index(drop=True)
        labels = [f"{int(row.n_segments_requested):,} / {row.compactness:.2f} / "
                  f"{row.graphcut_lambda:g}" for row in order.itertuples()]
        y = np.arange(len(order))
        ax.barh(y, order.water_IoU_vs_baseline_10m,
                color=ACADEMIC['soft_blue'], height=.7)
        ax.set_yticks(y, labels)
        ax.set_xlim(0, 1.02)
        ax.set_xlabel('Water-class IoU with reference parameterization')
        ax.set_ylabel('SLIC count / compactness / graph-cut λ')
        ax.set_title('SAR parameter stability (not accuracy)', loc='left', weight='bold')
        ax.axvline(1, color=ACADEMIC['navy'], lw=.75)
        fig.subplots_adjust(left=.34, right=.98, bottom=.19, top=.90)
        export_figure(fig, PUB_SUPP, 'FigS05_SAR_Parameter_Stability')
        register('Fig. S5', 'Supplementary',
            'Ten-metre candidate-water intersection over union between SAR parameter '
            'variants and the prespecified baseline; stability relative to baseline is not accuracy.')


## 9E. Submission index and output checks

The selection index identifies each proposed figure and intended location. Re-check all quantities against the actual run, review caption wording with the co-authors, and insert only the chosen outputs into the ODT files using LibreOffice; the manuscript's live Zotero fields must be retained. New independent reference data would be required to strengthen external accuracy claims.


In [ ]:
from PIL import Image

caption_table = pd.DataFrame(caption_records)
caption_table.to_csv(PUB_DIR/'Figure_Selection_And_Captions.csv', index=False)
# Verify TIFF exports and their recorded physical DPI.
for dest in (PUB_MAIN, PUB_SUPP):
    for path in sorted(dest.glob('*.tiff')):
        with Image.open(path) as im:
            dpi = im.info.get('dpi', (0, 0))
            assert abs(dpi[0]-600) <= 1 and abs(dpi[1]-600) <= 1, path

assert int(main_validation.evaluated_points.nunique()) == 1
assert int(confusion.sum()) == n_compared
assert int((ndfi_transition == 1).sum()) >= int((new_evidence == 1).sum())

figure_counts = {folder.name: len(list(folder.glob('*.tiff')))
                 for folder in (PUB_MAIN, PUB_SUPP)}
print('Version 2.1 | curated TIFF figure counts:', figure_counts)
print('Editable publication tables:', len(list(PUB_TABLES.glob('*.csv'))))
print('Selection index:', PUB_DIR/'Figure_Selection_And_Captions.csv')



## 10. Quality control and reproducibility record

The manifest lists all generated products. Reference-point counts are checked explicitly; GeoTIFF spatial resolution is distinct from printed figure DPI. Report any label-source limitations alongside validation statistics.


**Reproducibility note:** The JSON records the distinct sampling and GMM initialization seeds, the fitted mixture diagnostics, the constant SLIC/graph-cut scaling values, and baseline graph statistics. Cached segmentation and additional checks are not changes to the baseline classification method.


In [ ]:
# NoData and spatial-grid checks for all scientific GeoTIFF outputs.
spatial_outputs = (sar_output,ndfi_output,fusion_output,transition_output,
                   evidence_output,strict_output,ndfi_delta_output)
with rasterio.open(sar_output_10m) as ds:
    assert ds.crs==sar_crs and ds.transform==transform_10m
    assert (ds.height,ds.width)==(2000,2000) and ds.nodata==225
for path in spatial_outputs:
    with rasterio.open(path) as ds:
        assert ds.crs==sar_crs and ds.transform==transform_20m
        assert (ds.height,ds.width)==(1000,1000)
assert int(fusion_areas.pixels.sum())==1_000_000
assert int(transition_areas.pixels.sum())==1_000_000
assert int(evidence_areas.pixels.sum())==1_000_000
assert int((new_evidence==1).sum())==int((optical_new & sar_water & valid_evidence).sum())
assert len(samples)==len(references)
assert int(main_validation.evaluated_points.nunique())==int(1)

input_inventory=[]
for path in INPUT_FILES:
    if path.suffix=='.tif':
        with rasterio.open(path) as ds:
            meta=dict(bands=ds.count,width=ds.width,height=ds.height,
                      crs=str(ds.crs),resolution_m=list(ds.res),nodata=ds.nodata)
    else:
        meta=dict(features=len(references),crs=str(references.crs),geometry='Point')
    input_inventory.append(dict(path=str(path.relative_to(PROJECT_DIR)),**meta))
(REPRO_DIR/'Input_Inventory.json').write_text(
    json.dumps(input_inventory,indent=2,default=str),encoding='utf-8')

package_names=('numpy','scipy','scikit-learn','scikit-image','rasterio',
               'geopandas','pandas','matplotlib','Pillow')
package_versions={p:metadata.version(p) for p in package_names}
provenance=dict(
    date_utc=datetime.now(timezone.utc).isoformat(),
    study_area='20 km x 20 km Sutlej 2025',crs=str(sar_crs),
    dry_vv_used_for_training=False, wet_vv_source=str(SAR_WET.relative_to(PROJECT_DIR)),
    reference_source=str(REFERENCE_SHP.relative_to(PROJECT_DIR)),
    reference_status='Proxy reference labels; origin/independence to be described by authors',
    reference_used_for_training_or_tuning=False,
    gmm_components=2, gmm_samples_requested=GMM_SAMPLES,
    gmm_samples_actual=int(len(training)),
    gmm_sampling_seed=SEED, gmm_fit_random_state=GMM_FIT_RANDOM_STATE,
    gmm_n_init=GMM_N_INIT, gmm_max_iter=GMM_MAX_ITER,
    gmm_fit_diagnostics=gmm_diagnostics,
    gmm_reg_covar=GMM_REG_COVAR, gmm_bright_cap_db=GMM_BRIGHT_CAP_DB,
    gmm_log_odds_limit=GMM_LOG_ODDS_LIMIT,
    slic_intensity_db_range=[SLIC_MIN_DB, SLIC_MAX_DB],
    slic_max_num_iter=SLIC_MAX_ITER,
    graphcut_pairwise_sigma_db=GRAPHCUT_EDGE_SIGMA_DB,
    graphcut_capacity_scale=GRAPHCUT_CAPACITY_SCALE,
    graphcut_baseline_diagnostics=sar_info,
    sar_parameters=SELECTED, water_votes_per_2x2=WATER_VOTES_PER_2X2,
    ndfi_wet_water='wet >= 0',ndfi_new_water='dry < 0 and wet >= 0',
    figure_dpi=FIG_DPI,figure_nominal_width_mm=FIG_WIDTH_MM,
    intermethod_cohens_kappa=float(kappa),
    validation_jointly_valid_points=int(shared_available.sum()),
    software=package_versions)
(REPRO_DIR/'Analysis_Provenance.json').write_text(
    json.dumps(provenance,indent=2),encoding='utf-8')

manifest=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if not path.is_file(): continue
    row=dict(path=str(path.relative_to(PROJECT_DIR)),bytes=path.stat().st_size)
    if path.suffix=='.tiff':
        with Image.open(path) as image:
            row.update(width_px=image.width,height_px=image.height,
                       dpi=list(image.info.get('dpi',(None,None))))
    manifest.append(row)
pd.DataFrame(manifest).to_csv(REPRO_DIR/'Output_Manifest.csv',index=False)

print('Completed. Results in:',OUTPUT_ROOT)
print('SAR–NDFI inter-method κ:',round(kappa,4),'(not reference accuracy)')
print('Reference points jointly evaluated:',int(shared_available.sum()),'/',len(references))
print('Main manuscript figures:',len(list(MAIN_FIG_DIR.glob('*.tiff'))))
